# 08.6e YuE 完整歌曲生成入口

YuE 更接近一个完整歌曲生成工程。本 Notebook 构造官方 `infer.py` 命令；`CHAPTER08_YUE_REPO` 指向本地 YuE 仓库且依赖完整时会直接执行。


## 运行环境与安装

在 Jupyter 中选择 kernel：`Python 3.10 (chapter08-yue)`。YuE 是完整歌曲生成工程，不适合塞进本章共享环境；这里把环境建在 `CODE/venv_ch08_yue`，把官方仓库放在 `CODE/external/YuE`，便于统一管理和删除。长音频生成需要较大的 GPU 显存。官方推理脚本面向 CUDA 与 FlashAttention2；Linux/CUDA 或 WSL2/CUDA 是本 Notebook 的完整生成路径。没有 CUDA 的 macOS、Windows 或 CPU-only 环境可以完成下载和命令检查，但不会自动启动完整生成。

### macOS/Apple Silicon 说明

Mac 上的 MPS 不是 CUDA，也不能安装真正的 `flash-attn`。YuE 官方 `infer.py` 在 Stage 1 和 Stage 2 加载语言模型时使用 `attn_implementation="flash_attention_2"`，这是 CUDA/FlashAttention2 路线。即使 Apple Silicon 机器有较大的统一内存，也不能替代 CUDA 显存和 CUDA attention kernel。

因此，在 Mac 上看到 Stage 1、Stage 2 和 xcodec 权重都已下载，并不代表 YuE 可以本地完整生成。本 Notebook 在 Mac 上只做三件事：检查本地资产、展示 genre/lyrics 条件输入、生成可复制到 CUDA 机器运行的官方 `infer.py` 命令。完整音频生成请使用 Linux/CUDA、WSL2/CUDA 或远程 CUDA 机器。

Hugging Face 当前 CLI 命令名为 `hf`。若 `hf --help` 不可用，可按官方文档先安装 standalone CLI；mac/Linux 使用 `curl -LsSf https://hf.co/cli/install.sh | bash`，Windows PowerShell 使用 `powershell -ExecutionPolicy ByPass -c "irm https://hf.co/cli/install.ps1 | iex"`。YuE 需要下载四类本地资产：

- YuE 官方代码仓库：`external/YuE`
- xcodec/vocoder 权重：`external/YuE/inference/xcodec_mini_infer`
- Stage 1 歌曲 token 生成模型：`chapter08/models/m_a_p_yue_s1_7b_anneal_en_cot`
- Stage 2 token 上采样模型：`chapter08/models/m_a_p_yue_s2_1b_general`

只有 `external/YuE` 并不表示模型权重已经下载。若 `infer.py` 输出停在 `Fetching 3 files: 0%`，通常是 Stage 1 的 3 个 safetensors 分片还没有完整下载。请先用下面的 `hf download ... --local-dir ...` 命令把 checkpoint 放到 `chapter08/models`，再运行生成 cell。

Linux/WSL2 CUDA 终端使用：

```bash
cd CODE
python3.10 -m venv venv_ch08_yue
source venv_ch08_yue/bin/activate
python -m pip install --upgrade pip setuptools wheel
mkdir -p external
git clone https://github.com/multimodal-art-projection/YuE.git external/YuE
python -m pip install -r external/YuE/requirements.txt
python -m pip install pandas
python -m pip install flash-attn --no-build-isolation
python -m ipykernel install --user --name chapter08-yue --display-name "Python 3.10 (chapter08-yue)"
hf download m-a-p/xcodec_mini_infer --local-dir external/YuE/inference/xcodec_mini_infer
hf download m-a-p/YuE-s1-7B-anneal-en-cot --local-dir chapter08/models/m_a_p_yue_s1_7b_anneal_en_cot
hf download m-a-p/YuE-s2-1B-general --local-dir chapter08/models/m_a_p_yue_s2_1b_general
```

Windows PowerShell 可以准备同样的本地代码和权重；若要实际生成，建议在 WSL2/CUDA 或远程 Linux/CUDA 环境运行上面的命令：

```powershell
cd CODE
py -3.10 -m venv venv_ch08_yue
.\venv_ch08_yue\Scripts\Activate.ps1
python -m pip install --upgrade pip setuptools wheel
New-Item -ItemType Directory -Force external
git clone https://github.com/multimodal-art-projection/YuE.git external/YuE
python -m pip install -r external/YuE/requirements.txt
python -m pip install pandas
python -m ipykernel install --user --name chapter08-yue --display-name "Python 3.10 (chapter08-yue)"
hf download m-a-p/xcodec_mini_infer --local-dir external/YuE/inference/xcodec_mini_infer
hf download m-a-p/YuE-s1-7B-anneal-en-cot --local-dir chapter08/models/m_a_p_yue_s1_7b_anneal_en_cot
hf download m-a-p/YuE-s2-1B-general --local-dir chapter08/models/m_a_p_yue_s2_1b_general
```

如果 checkpoint 页面要求接受许可或登录账号，先在 Hugging Face 网页接受对应模型条款，再执行 `hf auth login` 后重新运行下载命令。

本 Notebook 会先读取 `CHAPTER08_YUE_REPO`，没有设置时自动检查 `CODE/external/YuE/inference`。如果你把 YuE 仓库放在其他位置，再设置 YuE 推理目录：

```bash
export CHAPTER08_YUE_REPO="$PWD/external/YuE/inference"
```

Windows PowerShell 对应为：

```powershell
$env:CHAPTER08_YUE_REPO = (Resolve-Path "external/YuE/inference").Path
```

如果官方 `requirements.txt` 在某台机器上与 Python 3.10 的 wheel 组合冲突，就删除 `CODE/venv_ch08_yue` 后用 Python 3.8 重建同名环境，再执行同一组 pip 和 git 命令。这个分支仍然可以放在 `CODE/venv_ch08_yue`，不需要使用全局 conda 环境。

如果状态表显示 `cuda=False` 或缺少 `flash_attn`，Notebook 会停在安装/硬件提示，不会启动官方 `infer.py`。这不是模型下载问题；YuE 的 Stage 1/Stage 2 权重可以在本地完整存在，但官方完整歌曲推理仍需要 CUDA/FlashAttention2 才是可用路径。


In [ ]:
from pathlib import Path
import os
import sys

# 路径推断：从 cwd 向上找含 CODE/chapter08/_common 的目录；ROOT 指向 CODE/chapter08/
_p = Path.cwd()
while not (_p / "CODE" / "chapter08" / "_common").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/chapter08/_common 的目录），请在项目内运行本 Notebook")
    _p = _parent
ROOT = _p / "CODE" / "chapter08"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Audio, display

from _common.config import load_yaml_config
from _common.device_utils import choose_device
from _common.paths import portable_path
from evaluation.comparison_table import append_model_comparison
from model_runners.base import GenerationRequest
from model_runners.conditioning import build_conditioning_rows

OUTPUT_TABLES = ROOT / "outputs" / "tables"
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)

def rel(path):
    return portable_path(path, ROOT)

def resolve_device(config=None):
    requested = os.getenv("CHAPTER08_DEVICE")
    if requested is None and config is not None:
        requested = str(config.get("device", "auto"))
    return choose_device(requested or "auto")

def print_setup_guidance(status):
    print(status.reason)
    if status.next_action:
        print(status.next_action)
    print("After completing the setup or moving to compatible hardware, rerun this Notebook; it will load and run the model directly.")

def print_runtime_guidance(error):
    print(str(error))
    print("Resolve the message above, then rerun this Notebook or the current cell.")

from model_runners.yue import YuERunner

runner = YuERunner()
status = runner.check_environment()
config = load_yaml_config(ROOT / "configs" / "yue_inference.yaml")
display(pd.DataFrame([status.as_row()]))


In [ ]:
repo_path = runner.repo_path()
download_rows = [
    {
        "asset": "YuE inference code",
        "local_path": rel(repo_path) if repo_path else "",
        "ready": bool(repo_path and (repo_path / "infer.py").exists()),
        "source": "https://github.com/multimodal-art-projection/YuE",
    },
    {
        "asset": "xcodec/vocoder weights",
        "local_path": rel((repo_path or ROOT) / "xcodec_mini_infer"),
        "ready": bool(repo_path and not [p for p in runner.required_xcodec_files if not (repo_path / p).exists()]),
        "source": "m-a-p/xcodec_mini_infer",
    },
    {
        "asset": "YuE Stage 1 checkpoint",
        "local_path": config["stage1_model_dir"],
        "ready": runner.model_dir_ready(ROOT / config["stage1_model_dir"]),
        "source": config["stage1_model_id"],
    },
    {
        "asset": "YuE Stage 2 checkpoint",
        "local_path": config["stage2_model_dir"],
        "ready": runner.model_dir_ready(ROOT / config["stage2_model_dir"]),
        "source": config["stage2_model_id"],
    },
]
display(pd.DataFrame(download_rows))
if not all(row["ready"] for row in download_rows):
    print("先按上方安装区块下载所有未就绪资产。下载完成后重启或重跑本 Notebook。")

display(pd.DataFrame(build_conditioning_rows("yue")))
for label, path in [("genre", ROOT / config["genre_txt"]), ("lyrics", ROOT / config["lyrics_file"])]:
    print(label, "file:", rel(path))
    if path.exists():
        print(path.read_text(encoding="utf-8").strip())


In [ ]:
request = GenerationRequest(
    prompt="",
    prompt_id="yue_demo",
    duration_sec=0,
    output_dir=ROOT / config["outputs"]["audio_dir"],
    extra={
        "command_template": config["command_template"],
        "genre_txt": config["genre_txt"],
        "lyrics_file": config["lyrics_file"],
        "stage1_model": config["stage1_model_dir"],
        "stage2_model": config["stage2_model_dir"],
        "run_external": status.available,
    },
)
command = runner.build_command(request)
print("YuE command:")
print(command.shell_command())
print("cwd:", rel(command.cwd) if command.cwd else "")

if status.available:
    result = runner.generate(request)
    append_model_comparison(
        ROOT / config["outputs"]["table_csv"],
        {
            "model_name": result.model_name,
            "prompt_id": result.prompt_id,
            "dataset_context": "genre and lyrics text files",
            "duration_sec": result.duration_sec,
            "wall_time_sec": result.wall_time_sec,
            "device": "",
            "output_audio_path": result.output_audio_path,
        },
    )
else:
    print_setup_guidance(status)
